In [ ]:
import numpy as np


def compute_window_metrics(traj, scores_df):
    """
    For each time window in scores_df, compute:
      - bbox area (deg^2, or projected units)
      - bbox perimeter
      - trajectory length (sum of segment distances in same units as lat/lon diffs)
    Returns scores_df with extra columns: bbox_area, bbox_perim, traj_len.
    """

    traj = traj.sort_values("timestamp").reset_index(drop=True)
    metrics = []

    for _, row in scores_df.iterrows():
        w_start, w_end = row["window_start"], row["window_end"]
        mask = (traj["timestamp"] >= w_start) & (traj["timestamp"] < w_end)
        df = traj[mask]

        if len(df) < 2:
            bbox_area = 0.0
            bbox_perim = 0.0
            traj_len = 0.0
        else:
            # axis‑aligned bounding box in lat‑lon space
            min_lat, max_lat = df["lat"].min(), df["lat"].max()
            min_lon, max_lon = df["lon"].min(), df["lon"].max()

            width = max_lon - min_lon
            height = max_lat - min_lat

            bbox_area = width * height
            bbox_perim = 2 * (width + height)

            # simple Euclidean length in lat‑lon units
            dx = np.diff(df["lon"].values)
            dy = np.diff(df["lat"].values)
            traj_len = np.sum(np.sqrt(dx**2 + dy**2))

        metrics.append((bbox_area, bbox_perim, traj_len))

    scores_df = scores_df.copy()
    scores_df[["bbox_area", "bbox_perim", "traj_len"]] = metrics
    return scores_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_window_metrics(scores_df, use_time=True):
    """
    Plot bbox area, perimeter, and trajectory length of each time window
    in three aligned subplots.
    """

    scores_df = scores_df.sort_values("window_start").reset_index(drop=True)

    if use_time:
        x = scores_df["window_start"]
        x_label = "Window start time"
    else:
        x = np.arange(len(scores_df))
        x_label = "Window index"

    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

    ax1, ax2, ax3 = axes

    # 1) Bounding box area
    ax1.plot(x, scores_df["bbox_area"], marker="o", color="tab:blue")
    ax1.set_ylabel("BBox area")
    ax1.set_title("Bounding box area per time window")

    # 2) Bounding box perimeter
    ax2.plot(x, scores_df["bbox_perim"], marker="o", color="tab:green")
    ax2.set_ylabel("BBox perimeter")
    ax2.set_title("Bounding box perimeter per time window")

    # 3) Trajectory length
    ax3.plot(x, scores_df["traj_len"], marker="o", color="tab:red")
    ax3.set_ylabel("Trajectory length")
    ax3.set_xlabel(x_label)
    ax3.set_title("Trajectory length per time window")

    if use_time:
        ax3.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
        fig.autofmt_xdate()

    plt.tight_layout()
    plt.show()

In [ ]:
scores_with_metrics = compute_window_metrics(traj, scores_df)
plot_window_metrics(scores_with_metrics, use_time=True)

NameError: name 'traj' is not defined